In [1]:
import os

In [2]:
%pwd

'c:\\Users\\kaush\\Desktop\\PROJECTS\\Chicken-disease-classification\\chicken-disease-classification-project-end2end\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\kaush\\Desktop\\PROJECTS\\Chicken-disease-classification\\chicken-disease-classification-project-end2end'

In [5]:
import tensorflow as tf
from pathlib import Path


In [17]:
def build_model(input_shape, num_classes, learning_rate):
    base_model = tf.keras.applications.ConvNeXtTiny(
        input_shape=input_shape,
        weights=None,
        include_top=False
    )

    for layer in base_model.layers:
        layer.trainable = False

    x = tf.keras.layers.Flatten()(base_model.output)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs=base_model.input, outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [18]:
from cnnClassifier.utils.common import read_yaml, create_directories, save_json
from cnnClassifier.constants import *

params = read_yaml(PARAMS_FILE_PATH)
config =  read_yaml(CONFIG_FILE_PATH)

[2026-06-23 03:57:33,755: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-23 03:57:33,763: INFO: common: yaml file: config\config.yaml loaded successfully]


In [11]:
model = build_model(
    input_shape=params.IMAGE_SIZE,
    num_classes=params.CLASSES,
    learning_rate=params.LEARNING_RATE
)

model.load_weights(config.training.training_model_path)

In [12]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [22]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model= self.config.training.training_model_path,
            training_data=  Path(self.config.data_ingestion.unzip_dir) / "Chicken-fecal-images",
            all_params= self.params,
            params_image_size= self.params.IMAGE_SIZE,
            params_batch_size= self.params.BATCH_SIZE
        )
        return eval_config

In [19]:
from urllib.parse import urlparse

In [20]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)

    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)


In [23]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()

except Exception as e:
   raise e

[2026-06-23 04:02:00,730: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-23 04:02:00,734: INFO: common: yaml file: params.yaml loaded successfully]


ValueError: No model config found in the file at artifacts/training/model.weights.h5.